<a href="https://colab.research.google.com/github/cbadenes/curso-pln/blob/main/notebooks/08_RAG_Basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG con generación real usando TinyLlama

Este notebook extiende la búsqueda dispersa y densa incorporando un modelo LLM real para generar respuestas.

Usaremos **TinyLlama-1.1B-Chat** para mantenerlo ligero y didáctico.

## 1. Dataset de películas

In [ ]:
documents = [
    "Inception es una película de ciencia ficción sobre sueños dentro de sueños.",
    "Interstellar trata sobre viajes espaciales y agujeros negros.",
    "The Dark Knight es una película de superhéroes con el Joker como villano.",
    "La La Land es un musical romántico ambientado en Los Ángeles.",
    "The Matrix explora la realidad simulada y la inteligencia artificial."
]

## 2. Búsqueda dispersa (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

def sparse_search(query, top_k=2):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_vectors)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    return [documents[i] for i in top_idx]

## 3. Búsqueda densa (embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(documents)

def dense_search(query, top_k=2):
    query_emb = embed_model.encode([query])[0]
    sims = doc_embeddings @ query_emb
    top_idx = sims.argsort()[::-1][:top_k]
    return [documents[i] for i in top_idx]

## 4. Cargar TinyLlama

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

## 5. Generador RAG con prompt

In [ ]:
def generate_answer(query, context_docs):
    context = "\n".join(context_docs)

    prompt = f"""<|system|>
Eres un experto en cine que proporciona recomendaciones y análisis de películas.
Usa el siguiente contexto para responder la pregunta del usuario.

Contexto:
{context}

<|user|>
{query}

<|assistant|>
Basándome en las películas mencionadas, recomiendo: """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        num_return_sequences=1,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

## 6. Pipeline RAG

In [ ]:
def rag_pipeline(query, search_fn):
    docs = search_fn(query)
    answer = generate_answer(query, docs)
    return docs, answer

## 7. Ejemplo

In [ ]:
query = "Recomiéndame una película sobre sueños o realidades complejas"

docs_sparse, answer_sparse = rag_pipeline(query, sparse_search)
docs_dense, answer_dense = rag_pipeline(query, dense_search)

print("=== CONTEXTO (Sparse) ===")
print(docs_sparse)
print("\n=== RESPUESTA ===")
print(answer_sparse)

print("\n\n=== CONTEXTO (Dense) ===")
print(docs_dense)
print("\n=== RESPUESTA ===")
print(answer_dense)

## 8. Discusión

- Ahora el modelo **sí genera texto real**
- El contexto recuperado influye directamente en la respuesta
- Puedes comparar cómo cambia la respuesta según el tipo de búsqueda